# 5. 자연어처리이론 실습

본 실습에서는 텍스트 전처리, 정수·원-핫 인코딩, 패딩, NNLM, Word2Vec, GloVe, FastText, BPE, WordPiece, SentencePiece의 핵심 원리를 코드로 확인합니다.


## 실습 목표

- 전처리 단계가 텍스트와 vocabulary에 미치는 영향을 설명할 수 있다.
- 문장 경계를 보존하여 Skip-gram 학습 쌍을 생성할 수 있다.
- Full softmax와 Negative Sampling의 차이를 코드 수준에서 설명할 수 있다.
- Word2Vec, GloVe, FastText의 학습 관점과 OOV 처리 차이를 비교할 수 있다.
- BPE, WordPiece, SentencePiece의 토큰화 결과를 비교할 수 있다.

> **실행 권장:** Colab의 새 런타임에서 위에서 아래로 순서대로 실행합니다. 다운로드가 포함된 셀은 네트워크 상태에 따라 시간이 소요될 수 있습니다.


## 5.1. Text preprocessing

### Setting

In [ ]:
# 수업 환경에 필요한 패키지 설치
!pip -q install nltk konlpy gensim sentencepiece transformers lxml

import random
import numpy as np
import torch
import nltk

# 재현 가능한 실습 결과를 위한 시드 고정
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# NLTK 3.9 계열을 포함한 최신 환경에서 필요한 리소스
nltk_resources = [
    'punkt',
    'punkt_tab',
    'averaged_perceptron_tagger',
    'averaged_perceptron_tagger_eng',
    'wordnet',
    'omw-1.4',
    'stopwords',
]
for resource in nltk_resources:
    nltk.download(resource, quiet=True)

print('실습 환경 설정 완료')


### Tokenization

단어 토큰화

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.tokenize import WordPunctTokenizer
nltk.download('punkt')

sentence = "Hello, My name is Jackson. Nice to meet you! I'm lawyer."
print('단어 토큰화1 :',word_tokenize(sentence))
print('단어 토큰화2 :',WordPunctTokenizer().tokenize(sentence))

문장 토큰화

In [ ]:
from nltk.tokenize import sent_tokenize

text = "Ph.D. students are using new technology in the lab. They're making discoveries and changing their field of study in new ways."
print('문장 토큰화1 :',sent_tokenize(text))

품사 태깅

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag

text = "Hello, My name is Jackson. Nice to meet you! I'm lawyer."
tokenized_sentence = word_tokenize(text)

print('단어 토큰화 :',tokenized_sentence)
print('품사 태깅 :',pos_tag(tokenized_sentence))

한국어 형태소 분석

In [ ]:
from konlpy.tag import Okt

okt = Okt()

text = "올 여름 시원한 수박이 왔어요"
print('OKT 형태소 분석 :',okt.morphs(text))
print('OKT 품사 태깅 :',okt.pos(text))
print('OKT 명사 추출 :',okt.nouns(text))

In [ ]:
from konlpy.tag import Kkma

kkma = Kkma()

text = "올 여름 시원한 수박이 왔어요"
print('Kkma 형태소 분석 :',kkma.morphs(text))
print('Kkma 품사 태깅 :',kkma.pos(text))
print('Kkma 명사 추출 :',kkma.nouns(text))

### Cleaning and Normalization

정제와 정규화는 모든 기호를 무조건 삭제하는 과정이 아닙니다. 분석 목적에 따라 URL, 이메일, 숫자, 이모지, 부정 표현 등을 보존하거나 특수 토큰으로 치환해야 합니다.


### 정규표현식(Regular Expression) 기초

정규표현식은 **문자열에서 특정한 패턴을 찾거나 치환하기 위한 표현 방법**입니다. Python에서는 `re` 모듈을 사용하며, 아래 실습에서는 `re.sub(패턴, 바꿀 문자열, 원문)`으로 HTML 태그, URL, 이메일, 반복 문장부호와 불필요한 공백을 처리합니다.

| 표현 | 의미 | 아래 코드에서의 활용 |
|---|---|---|
| `.` | 임의의 문자 1개 | URL·이메일 패턴의 일부 |
| `*`, `+` | 앞의 패턴이 각각 0회 이상, 1회 이상 반복 | `\S+`, `[^>]+` |
| `?` | 앞의 패턴이 0회 또는 1회 등장 | `https?`는 `http`와 `https` 모두 허용 |
| `[]` | 괄호 안 문자 중 하나 또는 문자 범위 | `[!?]`, `[A-Za-z]` |
| `[^...]` | 괄호 안 문자를 제외한 문자 | `<[^>]+>`로 HTML 태그 내부 탐색 |
| `\s`, `\S` | 공백 문자, 공백이 아닌 문자 | 연속 공백과 URL 탐색 |
| `|` | OR 조건 | URL 형식 두 가지 중 하나 탐색 |
| `()` | 패턴을 하나의 그룹으로 묶음 | 반복 문장부호 그룹 지정 |
| `\1` | 첫 번째 그룹에서 찾은 문자열을 다시 참조 | `!!!`을 `!`로 축약 |

> 정규표현식 앞에 붙는 `r`은 **raw string**을 의미합니다. 예를 들어 `r"\s+"`는 역슬래시를 Python의 이스케이프 문자로 먼저 처리하지 않고 정규표현식에 그대로 전달합니다.

정규표현식은 강력하지만 모든 텍스트를 무조건 삭제하는 용도로 사용하면 안 됩니다. 분석 목적에 따라 URL, 이메일, 숫자, 이모지와 문장부호를 제거할지 또는 특수 토큰으로 보존할지를 결정해야 합니다.


In [ ]:
import re
import unicodedata

sample_text = (
    "<p>새로운 온라인 강좌가 오늘 시작됩니다!!! 문의: info@example.org\n"
    "자세한 내용은 https://example.com/course 에서 확인하세요 😊</p>"
)


def normalize_text(text, lowercase=True):
    # 1. 유니코드 표기 통일
    text = unicodedata.normalize('NFKC', text)

    # 2. HTML 태그 제거
    text = re.sub(r'<[^>]+>', ' ', text)

    # 3. 과업에 유용할 수 있는 정보를 특수 토큰으로 보존
    text = re.sub(r'https?://\S+|www\.\S+', ' <URL> ', text)
    text = re.sub(r'[\w.%-]+@[\w.-]+\.[A-Za-z]{2,}', ' <EMAIL> ', text)

    # 4. 반복 문장부호와 공백 정규화
    text = re.sub(r'([!?])\1+', r'\1', text)
    text = re.sub(r'\s+', ' ', text).strip()

    if lowercase:
        text = text.lower()
    return text

print('[원문]')
print(sample_text)
print()
print('[정제·정규화 결과]')
print(normalize_text(sample_text))


#### Checkpoint

1. `<URL>`과 `<EMAIL>`을 삭제했을 때와 보존했을 때의 차이를 설명해 봅니다.  
2. 감성 분석에서는 이모지와 반복 문장부호를 무조건 제거해도 되는지 논의합니다.


### Lemmatization

In [ ]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

words = ['policies', 'doing', 'organizations', 'having', 'going', 'loved', 'living', 'flew', 'died', 'watching', 'has', 'started']

print('표제어 추출 전 :',words)
print('표제어 추출 후 :',[lemmatizer.lemmatize(word) for word in words])

In [ ]:
lemmatizer.lemmatize('dies', 'v')

In [ ]:
lemmatizer.lemmatize('has', 'v')

#### 품사 정보를 활용한 표제어 추출

WordNetLemmatizer는 기본적으로 명사로 처리하므로, 정확한 표제어를 얻으려면 품사 태깅 결과를 WordNet 품사로 변환해야 합니다.


In [ ]:
from nltk import pos_tag
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize


def to_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    if treebank_tag.startswith('V'):
        return wordnet.VERB
    if treebank_tag.startswith('N'):
        return wordnet.NOUN
    if treebank_tag.startswith('R'):
        return wordnet.ADV
    return wordnet.NOUN

sentence = "The students were studying and had completed their assignments."
tagged_words = pos_tag(word_tokenize(sentence))
pos_aware_lemmas = [
    lemmatizer.lemmatize(word, to_wordnet_pos(tag))
    for word, tag in tagged_words
]

print('품사 태깅 결과 :', tagged_words)
print('품사 기반 표제어:', pos_aware_lemmas)


### Stemming

In [ ]:
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

ps = PorterStemmer()

sentence = "Programmers program with programming languages"
words = word_tokenize(sentence)

stemmed_words = [ps.stem(word) for word in words]

print(stemmed_words)

In [ ]:
words = ['formalize', 'allowance', 'electricical']
print([ps.stem(word) for word in words])

In [ ]:
from nltk.stem import PorterStemmer
from nltk.stem import LancasterStemmer

ps = PorterStemmer()
ls = LancasterStemmer()

words = ['policy', 'doing', 'organization', 'have', 'going', 'love', 'lives', 'fly', 'dies', 'watched', 'has', 'starting']

print([ps.stem(word) for word in words])
print([ls.stem(word) for word in words])

### Stopword

불용어 확인

In [ ]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

stop_words = stopwords.words('english')
print(stop_words[:10])

불용어 제거

In [ ]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words('english'))
sentence = "This is an example showing off stop word filtration."
words = word_tokenize(sentence)

# 대소문자를 그대로 비교하면 'This'가 제거되지 않음
case_sensitive_result = [word for word in words if word not in stop_words]

# 소문자로 정규화하여 비교하되, 출력은 원래 토큰을 유지
normalized_result = [word for word in words if word.lower() not in stop_words]

print('대소문자 정규화 전:', case_sensitive_result)
print('대소문자 정규화 후:', normalized_result)


한국어 불용어

보편적으로 사용되는 한국어 불용어 리스트
https://www.ranks.nl/stopwords/korean

In [ ]:
from konlpy.tag import Okt

okt = Okt()
specific_stopwords = ['아버지', '방']  # 지정 불용어 설정

text = "아버지가 방에 들어가신다"
morphs = okt.morphs(text)  # 형태소 추출
filtered_sentence = [word for word in morphs if not word in specific_stopwords]

print(filtered_sentence)

### Integer Encoding

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

nltk.download('punkt') # 토크나이저 사용을 위한 데이터 다운로드

# 텍스트 데이터
text = "I love my dog. I love my cat."

# 토큰화
tokens = word_tokenize(text)

# 단어 빈도수 계산
freqdist = nltk.FreqDist(tokens)

# 단어를 빈도수에 따라 정수로 인코딩
int_encoding = {word: i for i, (word, _) in enumerate(freqdist.most_common(), 1)}

print(int_encoding)

### One-Hot Encoding

In [ ]:
# 원-핫 인코딩을 위한 함수
def one_hot_encoding(word, word2index):
    one_hot_vector = [0]*(len(word2index))
    index = word2index[word]
    one_hot_vector[index] = 1
    return one_hot_vector


In [ ]:
import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt')

text = "I love programming. I also love natural language processing."

# 토큰화
tokens = word_tokenize(text)

# 각 단어에 인덱스를 부여
word2index = {word: i for i, word in enumerate(sorted(set(tokens)))}

# 각 단어에 대해 원-핫 인코딩 수행
for word in tokens:
    print(f'{word}: {one_hot_encoding(word, word2index)}')

### Padding

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.probability import FreqDist
from collections import defaultdict


def pad_sequences(sentences, padding_token=0):
    max_length = max(len(sentence) for sentence in sentences)
    padded_sentences = [
        sentence + [padding_token] * (max_length - len(sentence))
        for sentence in sentences
    ]
    return padded_sentences


raw_sentences = [
    "I like playing soccer",
    "He enjoys reading books",
    "They are studying",
    "We are going to have a picnic",
]

tokenized_sentences = [word_tokenize(s) for s in raw_sentences]
fdist = FreqDist(word for sentence in tokenized_sentences for word in sentence)

# 0은 padding token을 위해 예약
word_to_id = defaultdict(lambda: 0)
for i, (word, _) in enumerate(fdist.most_common(), start=1):
    word_to_id[word] = i

encoded_sentences = [
    [word_to_id[word] for word in sentence]
    for sentence in tokenized_sentences
]
padded_sentences = pad_sequences(encoded_sentences, padding_token=0)

# 실제 토큰은 1, padding token은 0인 attention mask 생성
attention_masks = [
    [1 if token_id != 0 else 0 for token_id in sentence]
    for sentence in padded_sentences
]

for raw, encoded, padded, mask in zip(
    raw_sentences, encoded_sentences, padded_sentences, attention_masks
):
    print(f'문장: {raw}')
    print('정수 인코딩 :', encoded)
    print('패딩 결과   :', padded)
    print('Attention mask:', mask)
    print('-' * 60)


#### Checkpoint

패딩 값 `0`이 모델의 실제 입력으로 처리되지 않도록 attention mask가 필요한 이유를 설명해 봅니다.


## 5.2. Embedding

### NNLM PyTorch 구현

- 필요한 라이브러리

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

- 배치 생성 함수 정의

In [ ]:
def make_batch():
    input_batch = []
    target_batch = []

    # sentences = ["i like dog", "i love coffee", "i hate milk"]
    for sen in sentences:
        word = sen.split() # space tokenizer
        input = [word_dict[n] for n in word[:-1]] # create (1~n-1) as input
        target = word_dict[word[-1]] # create (n) as target, This is a causal language-modeling example

        input_batch.append(input)
        target_batch.append(target)

    return input_batch, target_batch

- 모델 클래스 정의

In [ ]:
class NNLM(nn.Module):
    def __init__(self):
        super(NNLM, self).__init__()
        self.C = nn.Embedding(n_class, m)
        self.H = nn.Linear(n_step * m, n_hidden, bias=False)
        self.d = nn.Parameter(torch.ones(n_hidden))
        self.U = nn.Linear(n_hidden, n_class, bias=False)
        self.W = nn.Linear(n_step * m, n_class, bias=False)
        self.b = nn.Parameter(torch.ones(n_class))

    def forward(self, X):
        X = self.C(X) # X : [batch_size, n_step, m]
        X = X.view(-1, n_step * m) # [batch_size, n_step * m]
        tanh = torch.tanh(self.d + self.H(X)) # [batch_size, n_hidden]
        output = self.b + self.W(X) + self.U(tanh) # [batch_size, n_class]
        return output

- 필요 파라미터와 데이터 설정

In [ ]:
n_step = 2
n_hidden = 2
m = 2

sentences = ["i like dog", "i love coffee", "i hate milk"]

- 단어 리스트 및 사전 생성

In [ ]:
word_list = sorted(set(" ".join(sentences).split()))

word_dict = {w: i for i, w in enumerate(word_list)}
number_dict = {i: w for i, w in enumerate(word_list)}
n_class = len(word_dict)

print(word_dict)
print(number_dict)


- 모델, 손실함수, 옵티마이저 설정

In [ ]:
model = NNLM()

# 손실함수와 최적화 함수 설정
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

- 학습 데이터를 배치 형태로 변환

In [ ]:
input_batch, target_batch = make_batch()
input_batch = torch.LongTensor(input_batch) # # [1, 0], [1, 2], [1, 4]
target_batch = torch.LongTensor(target_batch)

- 학습 진행

In [ ]:
# "i like", "i love", "i hate"
# torch.LongTensor([1, 0], [1, 2], [1, 4])

for epoch in range(5000):
    optimizer.zero_grad()
    output = model(input_batch)

    loss = criterion(output, target_batch)
    if (epoch + 1) % 1000 == 0:
        print('Epoch:', '%04d' % (epoch + 1), 'cost =', '{:.6f}'.format(loss.item()))

    loss.backward()
    optimizer.step()


- 모델 예측 및 테스트

In [ ]:
with torch.no_grad():
    predict = model(input_batch).argmax(dim=1)

print(
    [sen.split()[:2] for sen in sentences],
    '->',
    [number_dict[n.item()] for n in predict]
)


입력된 문장들("i like dog", "i love coffee", "i hate milk")에서 앞의 두 단어를 사용해 다음에 올 단어를 예측

### Word2Vec(Skip-gram) PyTorch 구현

- 필요한 라이브러리

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt


- 배치 생성 함수 정의

In [ ]:
def random_batch():
    random_inputs = []
    random_labels = []
    random_index = np.random.choice(range(len(skip_grams)), batch_size, replace=False)

    for i in random_index:
        random_inputs.append(np.eye(voc_size)[skip_grams[i][0]])  # target
        random_labels.append(skip_grams[i][1])  # context word

    return random_inputs, random_labels


- 모델 클래스 정의
  - \_\_init\_\_ 함수에서 필요한 레이어를 정의하고, forward 함수에서 이 레이어들을 어떻게 연결할지를 정의.

In [ ]:
class Word2Vec(nn.Module):
    def __init__(self):
        super(Word2Vec, self).__init__()
        # W and WT is not Traspose relationship
        self.W = nn.Linear(voc_size, embedding_size, bias=False) # voc_size > embedding_size Weight
        self.WT = nn.Linear(embedding_size, voc_size, bias=False) # embedding_size > voc_size Weight

    def forward(self, X):
        # X : [batch_size, voc_size]
        hidden_layer = self.W(X) # hidden_layer : [batch_size, embedding_size]
        output_layer = self.WT(hidden_layer) # output_layer : [batch_size, voc_size]
        return output_layer


- 필요 파라미터와 데이터 설정

In [ ]:
batch_size = 4
embedding_size = 64
window_size = 1

sentences = [
    "apple banana fruit",
    "banana orange fruit",
    "orange banana fruit",
    "dog cat animal",
    "cat monkey animal",
    "monkey dog animal",
]


- 단어 리스트 및 사전 생성

In [ ]:
word_list = sorted(set(" ".join(sentences).split()))
word_dict = {word: index for index, word in enumerate(word_list)}
number_dict = {index: word for word, index in word_dict.items()}
voc_size = len(word_list)

print('Vocabulary:', word_dict)


- Skip-gram 생성

In [ ]:
# 문장 경계를 보존하여 Skip-gram positive pair 생성
skip_grams = []

for sentence in sentences:
    tokens = sentence.split()
    for center_index, center_word in enumerate(tokens):
        left = max(0, center_index - window_size)
        right = min(len(tokens), center_index + window_size + 1)

        for context_index in range(left, right):
            if center_index == context_index:
                continue
            skip_grams.append([
                word_dict[center_word],
                word_dict[tokens[context_index]],
            ])

print('생성된 positive pair 수:', len(skip_grams))
print('앞의 12개 pair:')
for center_id, context_id in skip_grams[:12]:
    print(f'({number_dict[center_id]}, {number_dict[context_id]})')


#### Checkpoint: 문장 경계

모든 문장을 하나의 문자열로 연결한 뒤 window를 적용하면 앞 문장의 마지막 단어와 다음 문장의 첫 단어가 잘못된 문맥 쌍으로 생성됩니다. 위 코드는 문장별로 pair를 생성하여 이 문제를 방지합니다.


- 모델, 손실 함수, 옵티마이저 설정

In [ ]:
model = Word2Vec()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


- 모델 학습

In [ ]:
# Training
for epoch in range(5000):
    input_batch, target_batch = random_batch()
    input_batch = torch.Tensor(input_batch)
    target_batch = torch.LongTensor(target_batch)

    optimizer.zero_grad()
    output = model(input_batch)

    # output : [batch_size, voc_size], target_batch : [batch_size] (LongTensor, not one-hot)
    loss = criterion(output, target_batch)
    if (epoch + 1) % 500 == 0:
        print('Epoch:', '%04d' % (epoch + 1), 'cost =', '{:.6f}'.format(loss.item()))

    loss.backward()
    optimizer.step()


- 임베딩 결과 시각화

In [ ]:
from sklearn.decomposition import PCA

# nn.Linear의 W.weight shape: [embedding_size, vocabulary_size]
# 단어별 벡터를 얻기 위해 전치하여 [vocabulary_size, embedding_size]로 변환
embedding_matrix = model.W.weight.detach().cpu().numpy().T

pca = PCA(n_components=2, random_state=SEED)
coordinates = pca.fit_transform(embedding_matrix)

plt.figure(figsize=(8, 6))
for i, label in enumerate(word_list):
    x, y = coordinates[i]
    plt.scatter(x, y)
    plt.annotate(label, (x, y), xytext=(5, 3), textcoords='offset points')

plt.title('Skip-gram embeddings projected by PCA')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.grid(alpha=0.3)
plt.show()


시각화 결과에서, 서로 가까운 위치에 있는 단어들은 비슷한 문맥 또는 의미를 가진다는 것을 나타냄. 보다 정확한 워드 임베딩을 얻기 위해서는 더 많은 데이터와 복잡한 모델이 필요함.

### Negative Sampling을 사용한 Skip-gram(SGNS) 미니 실습

앞의 Skip-gram은 전체 vocabulary에 대해 softmax를 계산했습니다. SGNS는 관측된 중심 단어–주변 단어 쌍을 `1`, 무작위로 구성한 쌍을 `0`으로 두고 이진 분류 문제로 학습합니다.


In [ ]:
from collections import defaultdict

# 중심 단어별 실제 주변 단어 집합
positive_contexts = defaultdict(set)
for center_id, context_id in skip_grams:
    positive_contexts[center_id].add(context_id)

rng = np.random.default_rng(SEED)
positive_pairs = [(c, o, 1.0) for c, o in skip_grams]
negative_pairs = []

# 각 positive pair마다 실제 주변 단어가 아닌 negative context 하나를 샘플링
for center_id, _ in skip_grams:
    candidates = [
        word_id for word_id in range(voc_size)
        if word_id != center_id and word_id not in positive_contexts[center_id]
    ]
    negative_context = int(rng.choice(candidates))
    negative_pairs.append((center_id, negative_context, 0.0))

sgns_samples = positive_pairs + negative_pairs
rng.shuffle(sgns_samples)

print('center	context	label')
for center_id, context_id, label in sgns_samples[:12]:
    print(f'{number_dict[center_id]}	{number_dict[context_id]}	{int(label)}')


In [ ]:
class SGNS(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.center_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.context_embedding = nn.Embedding(vocab_size, embedding_dim)

    def forward(self, center_ids, context_ids):
        center_vectors = self.center_embedding(center_ids)
        context_vectors = self.context_embedding(context_ids)
        return (center_vectors * context_vectors).sum(dim=1)


center_ids = torch.tensor([sample[0] for sample in sgns_samples], dtype=torch.long)
context_ids = torch.tensor([sample[1] for sample in sgns_samples], dtype=torch.long)
labels = torch.tensor([sample[2] for sample in sgns_samples], dtype=torch.float32)

sgns_model = SGNS(voc_size, embedding_dim=16)
sgns_criterion = nn.BCEWithLogitsLoss()
sgns_optimizer = optim.Adam(sgns_model.parameters(), lr=0.03)

for epoch in range(1000):
    sgns_optimizer.zero_grad()
    logits = sgns_model(center_ids, context_ids)
    loss = sgns_criterion(logits, labels)
    loss.backward()
    sgns_optimizer.step()

    if (epoch + 1) % 200 == 0:
        print(f'Epoch {epoch + 1:4d} | loss={loss.item():.4f}')


In [ ]:
with torch.no_grad():
    probabilities = torch.sigmoid(sgns_model(center_ids, context_ids))

positive_mean = probabilities[labels == 1].mean().item()
negative_mean = probabilities[labels == 0].mean().item()

print(f'Positive pair 평균 예측 확률: {positive_mean:.3f}')
print(f'Negative pair 평균 예측 확률: {negative_mean:.3f}')


#### Checkpoint

Full softmax 방식은 모든 단어에 대한 점수를 계산하지만, Negative Sampling은 선택된 positive/negative pair에 대해서만 업데이트합니다. vocabulary가 매우 클 때 두 방식의 계산량 차이를 설명해 봅니다.


### gensim 라이브러리를 사용하여 Word2Vec 모델 학습 실습

- 훈련 데이터 다운

In [ ]:
import re
import urllib.request
import zipfile
from lxml import etree
from nltk.tokenize import word_tokenize, sent_tokenize

import nltk
nltk.download('punkt')

In [ ]:
urllib.request.urlretrieve("https://raw.githubusercontent.com/ukairia777/tensorflow-nlp-tutorial/main/09.%20Word%20Embedding/dataset/ted_en-20160408.xml", filename="ted_en-20160408.xml")

해당 데이터 파일은 xml 문법으로, 영어문장이 담긴 <content>와 </content> 사이의 내용을 전처리 해야함.

In [ ]:
with open('ted_en-20160408.xml', 'r', encoding='UTF8') as targetXML:
    target_text = etree.parse(targetXML)

# xml 파일로부터 <content>와 </content> 내용 파싱
parse_text = '\n'.join(target_text.xpath('//content/text()'))

# 정규 표현식의 sub 모듈을 통해 (Audio), (Laughter) 등의 배경음 제거.
content_text = re.sub(r'\([^)]*\)', '', parse_text)

# NLTK를 이용하여 문장 토큰화를 수행.
sent_text = sent_tokenize(content_text)

# 구두점을 제거 및 대문자를 소문자로 변환.
normalized_text = []
for string in sent_text:
     tokens = re.sub(r"[^a-z0-9]+", " ", string.lower())
     normalized_text.append(tokens)

# NLTK를 이용하여 단어 토큰화를 수행.
result = [word_tokenize(sentence) for sentence in normalized_text]

In [ ]:
print('총 샘플 개수 : {}'.format(len(result)))

In [ ]:
# 샘플 3개 출력
for line in result[:3]:
    print(line)

- Word2Vec 훈련
  - vector_size = 워드 벡터의 특징 값. 즉, 임베딩 된 벡터의 차원.
  - window = 컨텍스트 윈도우 크기
  - min_count = 단어 최소 빈도 수 제한 (빈도가 적은 단어들은 학습하지 않는다.)
  - workers = 학습을 위한 프로세스 수
  - sg = 0은 CBOW, 1은 Skip-gram.

In [ ]:
from gensim.models import Word2Vec
from gensim.models import KeyedVectors

wv_model = Word2Vec(
    sentences=result,
    vector_size=100,
    window=5,
    min_count=5,
    workers=1,   # 재현성을 위해 단일 worker 사용
    sg=0,
    seed=SEED,
)


- 가장 유사한 단어 출력

In [ ]:
model_result = wv_model.wv.most_similar("girl")
print(model_result)

- Word2Vec 모델 저장 & 로드

In [ ]:
wv_model.wv.save_word2vec_format('my_w2v') # 모델 저장
loaded_model = KeyedVectors.load_word2vec_format("my_w2v") # 모델 로드

In [ ]:
model_result = loaded_model.most_similar("man")
print(model_result)

### 사전학습 GloVe 모델 활용 실습

이 절에서는 GloVe를 새로 학습하지 않고, Wikipedia와 Gigaword로 사전학습된 임베딩을 내려받아 조회합니다. 동일 corpus에서 직접 학습한 실험과 구분해야 합니다.


In [ ]:
import gensim.downloader as api

# 수업 시간과 메모리를 고려해 50차원 모델 사용
# 최초 실행 시 모델 파일 다운로드가 필요함
gv_model = api.load('glove-wiki-gigaword-50')

king_vector = gv_model['king']
print('king vector shape:', king_vector.shape)
print('Words similar to king:', gv_model.most_similar('king'))


### FastText 모델 학습 실습


- Word2vec은 모르는 단어에 대해 임베딩 벡터가 존재하지 않아 단어의 유사도를 계산할 수 없음.

In [ ]:
wv_model = KeyedVectors.load_word2vec_format('my_w2v')

try:
    print(wv_model.most_similar('electrofishing'))
except KeyError as error:
    print('예상된 OOV 오류:', error)
    print('Word2Vec은 vocabulary에 없는 단어의 벡터를 직접 생성하지 못합니다.')


- FastText 학습

In [ ]:
from gensim.models import FastText

ft_model = FastText(
    sentences=result,
    vector_size=100,
    window=5,
    min_count=5,
    workers=1,
    sg=1,
    seed=SEED,
)


In [ ]:
ft_model.wv.most_similar("electrofishing")

- FastText는 유사한 단어를 계산하여 출력하는 것을 확인할 수 있음.

### 동일 corpus에서 학습한 Word2Vec과 FastText의 동작 특성 비교

아래 비교는 정량적 성능 평가가 아니라, 동일한 TED corpus에서 학습한 두 모델의 vocabulary와 OOV 처리 차이를 확인하는 실습입니다.


In [ ]:
import pandas as pd

def safe_most_similar(keyed_vectors, query, topn=3):
    try:
        results = keyed_vectors.most_similar(query, topn=topn)
        return ', '.join(f'{word}({score:.3f})' for word, score in results)
    except KeyError:
        return 'OOV: vector 없음'

queries = ['girl', 'electrofishing']
comparison_rows = []

for query in queries:
    comparison_rows.append({
        'query': query,
        'Word2Vec': safe_most_similar(wv_model, query),
        'FastText': safe_most_similar(ft_model.wv, query),
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df


#### 해석

- 두 모델 모두 corpus에 충분히 등장한 단어에는 문맥 기반 벡터를 제공합니다.
- Word2Vec은 vocabulary 밖의 단어에 대해 오류가 발생합니다.
- FastText는 character n-gram을 결합하므로 OOV 단어에도 벡터를 구성할 수 있습니다.
- 유사 단어 출력만으로 모델의 전체 성능을 단정해서는 안 되며, 실제 과업에서는 별도의 정량 평가가 필요합니다.


## 5.3. OOV problem

### BPE (Byte Pair Encoding) 구현


- 필요한 라이브러리

In [ ]:
import re, collections
from IPython.display import display, Markdown, Latex

- BPE 수행 횟수 지정

In [ ]:
num_merges = 10

- </w> 단어 맨 끝에 붙이는 특수 문자, 단어는 글자 단위 분리

In [ ]:
initial_dictionary = {
    'l o w </w>': 5,
    'l o w e r </w>': 2,
    'n e w e s t </w>': 6,
    'w i d e s t </w>': 3,
}


- 가장 빈도수가 높은 유니그램의 쌍을 하나의 유니그램으로 통합하는 과정으로 num_merges회 반복

In [ ]:
def get_stats(dictionary):
    # 유니그램의 pair들 빈도수 카운트
    pairs = collections.defaultdict(int)
    for word, freq in dictionary.items():
        symbols = word.split()
        for i in range(len(symbols)-1):
            pairs[symbols[i],symbols[i+1]] += freq
    print('현재 pair들의 빈도수 :', dict(pairs))
    return pairs


In [ ]:
def merge_dictionary(pair, v_in):
    v_out = {}
    bigram = re.escape(' '.join(pair))
    p = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    for word in v_in:
        w_out = p.sub(''.join(pair), word)
        v_out[w_out] = v_in[word]
    return v_out

In [ ]:
# 셀을 다시 실행해도 같은 결과가 나오도록 초기화
bpe_codes = {}
bpe_codes_reverse = {}
dictionary = initial_dictionary.copy()

for i in range(num_merges):
    display(Markdown('### Iteration {}'.format(i + 1)))
    pairs = get_stats(dictionary)
    if not pairs:
        break

    best = max(pairs, key=pairs.get)
    dictionary = merge_dictionary(best, dictionary)

    bpe_codes[best] = i
    bpe_codes_reverse[best[0] + best[1]] = best

    print('new merge:', best)
    print('dictionary:', dictionary)


- merge 기록 출력

In [ ]:
print(bpe_codes)

- OOV에 대처

In [ ]:
def get_pairs(word):
    """Return set of symbol pairs in a word.
    Word is represented as a tuple of symbols (symbols being variable-length strings).
    """
    pairs = set()
    prev_char = word[0]
    for char in word[1:]:
        pairs.add((prev_char, char))
        prev_char = char
    return pairs


def encode(orig):
    """Encode word based on list of BPE merge operations, which are applied consecutively"""

    word = tuple(orig) + ('</w>',)
    display(Markdown("__word split into characters:__ <tt>{}</tt>".format(word)))

    pairs = get_pairs(word)

    if not pairs:
        return orig

    iteration = 0
    while True:
        iteration += 1
        display(Markdown("__Iteration {}:__".format(iteration)))

        print("bigrams in the word: {}".format(pairs))
        bigram = min(pairs, key = lambda pair: bpe_codes.get(pair, float('inf')))
        print("candidate for merging: {}".format(bigram))
        if bigram not in bpe_codes:
            display(Markdown("__Candidate not in BPE merges, algorithm stops.__"))
            break
        first, second = bigram
        new_word = []
        i = 0
        while i < len(word):
            try:
                j = word.index(first, i)
                new_word.extend(word[i:j])
                i = j
            except:
                new_word.extend(word[i:])
                break

            if word[i] == first and i < len(word)-1 and word[i+1] == second:
                new_word.append(first+second)
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        new_word = tuple(new_word)
        word = new_word
        print("word after merging: {}".format(word))
        if len(word) == 1:
            break
        else:
            pairs = get_pairs(word)

    # </w>는 출력 X
    if word[-1] == '</w>':
        word = word[:-1]
    elif word[-1].endswith('</w>'):
        word = word[:-1] + (word[-1].replace('</w>',''),)

    return word

In [ ]:
encode("loki")

In [ ]:
encode("lowest")

### Sentencepiece

IMDB 리뷰 토큰화 하기

- sentencepiece 설치

In [ ]:
!pip install sentencepiece

- 필요한 라이브러리 및 데이터 로드

In [ ]:
import sentencepiece as spm
import pandas as pd
import urllib.request
import csv

In [ ]:
urllib.request.urlretrieve("https://raw.githubusercontent.com/LawrenceDuan/IMDb-Review-Analysis/master/IMDb_Reviews.csv", filename="IMDb_Reviews.csv")

In [ ]:
train_df = pd.read_csv('IMDb_Reviews.csv')
train_df = train_df.dropna(subset=['review']).copy()
train_df['review'] = train_df['review'].astype(str)
train_subset = train_df['review'].head(10000)
train_subset.head()


In [ ]:
print('전체 리뷰 개수:', len(train_df))
print('토크나이저 학습에 사용할 리뷰 개수:', len(train_subset))


- sentencepiece 입력으로 사용하기 위해 dataframe을 txt로 저장

In [ ]:
with open('imdb_review.txt', 'w', encoding='utf8') as f:
    f.write('\n'.join(train_subset))


- sentencepiece로 단어 집합과 각 단어에 고유한 정수를 부여
  - input : 학습시킬 파일
  - model_prefix : 만들어질 모델 이름
  - vocab_size : 단어 집합의 크기
  - model_type : 사용할 모델 (unigram(default), bpe, char, word)
  - max_sentence_length: 문장의 최대 길이
  - pad_id, pad_piece: pad token id, 값
  - unk_id, unk_piece: unknown token id, 값
  - bos_id, bos_piece: begin of sentence token id, 값
  - eos_id, eos_piece: end of sequence token id, 값
  - user_defined_symbols: 사용자 정의 토큰

In [ ]:
spm.SentencePieceTrainer.Train(
    input='imdb_review.txt',
    model_prefix='imdb',
    vocab_size=5000,
    model_type='bpe',
    max_sentence_length=9999,
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3,
    pad_piece='<pad>',
    unk_piece='<unk>',
    bos_piece='<s>',
    eos_piece='</s>',
    hard_vocab_limit=False,
)


- vocab 생성 완료 후, imdb.model 과 imdb.vocab 파일 두개가 생성 됨.

In [ ]:
vocab_list = pd.read_csv('imdb.vocab', sep='\t', header=None, quoting=csv.QUOTE_NONE)
vocab_list.sample(10)

In [ ]:
len(vocab_list) # 위 인자로, 단어 집합의 크기를 5000개로 제한

- model 파일을 로드하여, 단어 시퀀스를 정수 시퀀스로 바꾸는 인코딩, 디코딩 작업이 가능

In [ ]:
sp = spm.SentencePieceProcessor()
vocab_file = "imdb.model"
sp.load(vocab_file)

In [ ]:
special_pieces = ['<pad>', '<unk>', '<s>', '</s>']
for piece in special_pieces:
    print(f'{piece:6s} -> id {sp.PieceToId(piece)}')


### WordPiece와 SentencePiece 토큰화 비교

- **WordPiece**는 사전학습 모델이 보유한 고정 vocabulary를 사용합니다.
- **SentencePiece**는 원시 문장에서 직접 vocabulary를 학습할 수 있는 토크나이저 학습 프레임워크이며, 내부 모델로 BPE 또는 Unigram을 사용할 수 있습니다.
- 토큰화 결과는 알고리즘뿐 아니라 학습 corpus와 vocabulary 크기에 크게 좌우됩니다.


In [ ]:
from transformers import AutoTokenizer

# KLUE-BERT는 WordPiece 기반 한국어 토크나이저를 사용
wordpiece_tokenizer = AutoTokenizer.from_pretrained('klue/bert-base')

comparison_texts = [
    '온라인 강의에서 자연어처리를 공부합니다.',
    'electrofishing is unexpectedly difficult.',
]

for text in comparison_texts:
    print('원문:', text)
    print('WordPiece    :', wordpiece_tokenizer.tokenize(text))
    print('SentencePiece:', sp.encode(text, out_type=str))
    print('-' * 80)


#### Checkpoint

같은 문장을 서로 다른 토크나이저가 다르게 분해하는 이유를 `학습 corpus`, `vocabulary`, `알고리즘`의 세 관점에서 설명해 봅니다.


- encode_as_pieces: 문장을 입력하면 서브 워드 시퀀스로 변환
- encode_as_ids: 문장을 입력하면 정수 시퀀스로 변환

In [ ]:
lines = [
  "I didn't at all think of it this way.",
  "I have waited a long time for someone to film"
]
for line in lines:
  print(line)
  print(sp.encode_as_pieces(line))
  print(sp.encode_as_ids(line))
  print()

- GetPieceSize: 단어 집합의 크기를 확인

In [ ]:
sp.GetPieceSize()

- idToPiece: 정수로부터 맵핑되는 서브워드 반환

In [ ]:
sp.IdToPiece(430)

- PieceTold: 서브워드로부터 맵핑되는 정수로 변환

In [ ]:
sp.PieceToId('▁character')

- DecodeIds: 정수 시퀀스로부터 문장 변환

In [ ]:
sp.DecodeIds([41, 141, 1364, 1120, 4, 666, 285, 92, 1078, 33, 91])

- DecodePieces: 서브워드 시퀀스로부터 문장 변환

In [ ]:
sp.DecodePieces(['▁I', '▁have', '▁wa', 'ited', '▁a', '▁long', '▁time', '▁for', '▁someone', '▁to', '▁film'])

- encode: 인자값에 따라 정수 또는 서브워드 시퀀스로 변환

In [ ]:
print(sp.encode('I have waited a long time for someone to film', out_type=str))
print(sp.encode('I have waited a long time for someone to film', out_type=int))